# 🎵 Music Studio — Download · Identify · Complete
**Partie 1** — Téléchargement (Instagram / YouTube)  
**Partie 2** — Identification du titre (AudD.io) + Genre (HuggingFace)  
**Partie 3** — Génération des parties manquantes (→ 2min30)  
**Partie 4** — Interface graphique unifiée

---
## 📦 PARTIE 1 — Téléchargement

In [ ]:
# Cellule 1.1 — Installation Partie 1
import sys
!{sys.executable} -m pip install yt-dlp --quiet
print('✅ yt-dlp installé')

In [ ]:
# Cellule 1.2 — Imports & fonctions Partie 1
import yt_dlp, os, glob

def p1_download(url: str, fmt: str = 'mp3', cookies_path: str = '') -> dict:
    out_dir = './downloads'
    os.makedirs(out_dir, exist_ok=True)
    if fmt == 'mp4':
        ydl_opts = {
            'format': 'bestvideo+bestaudio/best',
            'outtmpl': f'{out_dir}/%(title)s.%(ext)s',
            'merge_output_format': 'mp4',
            'quiet': True, 'no_warnings': True,
        }
    else:
        ydl_opts = {
            'format': 'bestaudio/best',
            'outtmpl': f'{out_dir}/%(title)s.%(ext)s',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': fmt,
                'preferredquality': '320' if fmt == 'mp3' else '0',
            }],
            'quiet': True, 'no_warnings': True,
        }
    if cookies_path and os.path.exists(cookies_path):
        ydl_opts['cookiefile'] = cookies_path
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            title = info.get('title', 'audio')
        files_found = sorted(glob.glob(f'{out_dir}/*.{fmt}'), key=os.path.getmtime, reverse=True)
        if files_found:
            return {'success': True, 'path': files_found[0], 'title': title, 'error': ''}
        return {'success': False, 'path': '', 'title': '', 'error': 'Fichier introuvable après téléchargement'}
    except Exception as e:
        return {'success': False, 'path': '', 'title': '', 'error': str(e)}

def p1_get_audio_files():
    f = glob.glob('./downloads/*.mp3') + glob.glob('./downloads/*.wav')
    return sorted([os.path.basename(x) for x in f])

print('✅ Partie 1 prête')

---
## 🔍 PARTIE 2 — Identification (AudD.io + HuggingFace)

In [ ]:
# Cellule 2.1 — Installation Partie 2
!pip install transformers torch torchaudio requests --quiet
print('✅ Dépendances Partie 2 installées')

In [ ]:
# Cellule 2.2 — Imports & fonctions Partie 2
import torch
import torchaudio
import numpy as np
import requests
from transformers import pipeline

# ─── AudD.io — Reconnaissance du titre exact ───
def p2_identify_title(audio_path: str, audd_key: str) -> dict:
    if not audd_key:
        return {'success': False, 'title': '', 'artist': '', 'album': '', 'error': 'Pas de clé AudD fournie'}
    try:
        with open(audio_path, 'rb') as f:
            data = {'api_token': audd_key, 'return': 'apple_music,spotify'}
            result = requests.post('https://api.audd.io/', data=data, files={'file': f}, timeout=15).json()
        if result.get('status') == 'success' and result.get('result'):
            r = result['result']
            spotify = r.get('spotify') or {}
            apple = r.get('apple_music') or {}
            return {
                'success': True,
                'title': r.get('title', '?'),
                'artist': r.get('artist', '?'),
                'album': r.get('album', '?'),
                'release_date': r.get('release_date', '?'),
                'spotify_url': spotify.get('external_urls', {}).get('spotify', ''),
                'apple_url': apple.get('url', ''),
                'error': ''
            }
        return {'success': False, 'title': '', 'artist': '', 'album': '',
                'error': result.get('error', {}).get('error_message', 'Titre non trouvé')}
    except Exception as e:
        return {'success': False, 'title': '', 'artist': '', 'album': '', 'error': str(e)}


# ─── HuggingFace — Détection du genre (2 modèles) ───
P2_MODELS = [
    {
        'id': 'laion/clap-htsat-unfused',
        'name': 'CLAP (LAION)',
        'type': 'zero-shot-audio-classification',
        'description': 'Modèle contrastif audio-texte, très bon pour la classification de genre',
        'labels': [
            'pop music', 'hip hop', 'electronic music', 'rock music', 'jazz',
            'classical music', 'r&b soul', 'reggaeton', 'latin music', 'house music',
            'techno', 'trap music', 'lo-fi music', 'ambient music', 'funk',
            'metal', 'indie music', 'dance music', 'afrobeats', 'drill music'
        ]
    },
    {
        'id': 'MIT/ast-finetuned-audioset-10-10-0.4593',
        'name': 'AST (MIT AudioSet)',
        'type': 'audio-classification',
        'description': 'Audio Spectrogram Transformer entraîné sur AudioSet (527 classes)'
    }
]

def p2_load_audio_for_hf(path: str, target_sr: int = 16000) -> np.ndarray:
    waveform, sr = torchaudio.load(path)
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform[:, :target_sr * 30]
    return waveform.squeeze().numpy()

def p2_run_model(model_info: dict, audio_path: str) -> dict:
    try:
        audio_np = p2_load_audio_for_hf(audio_path)
        clf = pipeline(model_info['type'], model=model_info['id'])
        if model_info['type'] == 'zero-shot-audio-classification':
            results = clf(audio_np, candidate_labels=model_info['labels'], sampling_rate=16000)
        else:
            results = clf(audio_np, sampling_rate=16000, top_k=5)
        return {
            'success': True,
            'model_name': model_info['name'],
            'description': model_info['description'],
            'top_results': [{'label': r['label'], 'score': r['score']} for r in results[:5]],
            'error': ''
        }
    except Exception as e:
        return {
            'success': False,
            'model_name': model_info['name'],
            'description': model_info.get('description', ''),
            'top_results': [],
            'error': str(e)
        }

def p2_identify_all(audio_path: str, audd_key: str = '', progress_cb=None) -> dict:
    total = 1 + len(P2_MODELS)
    if progress_cb: progress_cb(0, total, '⏳ Identification du titre via AudD.io...')
    title_result = p2_identify_title(audio_path, audd_key)
    if progress_cb: progress_cb(1, total, '✅ Titre traité — analyse des genres...')
    genre_results = []
    for i, model_info in enumerate(P2_MODELS):
        if progress_cb:
            progress_cb(1 + i, total, f'⏳ Modèle {i+1}/{len(P2_MODELS)} : {model_info["name"]}...')
        genre_results.append(p2_run_model(model_info, audio_path))
    if progress_cb: progress_cb(total, total, '✅ Analyse complète !')
    return {'title': title_result, 'genres': genre_results}

print('✅ Partie 2 prête —', len(P2_MODELS), 'modèles genre + AudD.io pour le titre')

---
## 🎼 PARTIE 3 — Complétion musicale (→ 2min30)

In [ ]:
# Cellule 3.1 — Installation Partie 3
!pip install transformers accelerate scipy torchaudio --quiet
print('✅ Dépendances Partie 3 installées')

In [ ]:
# Cellule 3.2 — Imports & chargement modèle (small par défaut)
import torch
import torchaudio
import scipy.io.wavfile
import numpy as np
from transformers import AutoProcessor, MusicgenForConditionalGeneration

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Device : {device}')
if device == 'cpu':
    print('⚠️ Pas de GPU ! Va dans Exécution → Modifier le type d\'exécution → GPU T4')

P3_MODEL_NAME = 'facebook/musicgen-small'
p3_processor = None
p3_model = None

def p3_load_model():
    global p3_processor, p3_model
    if p3_model is not None:
        return True
    try:
        print(f'⏳ Chargement {P3_MODEL_NAME}...')
        p3_processor = AutoProcessor.from_pretrained(P3_MODEL_NAME)
        p3_model = MusicgenForConditionalGeneration.from_pretrained(P3_MODEL_NAME).to(device)
        print(f'✅ Modèle chargé sur {device.upper()}')
        return True
    except Exception as e:
        print(f'❌ Erreur chargement modèle : {e}')
        return False

print('✅ Partie 3 prête (modèle chargé à la première génération)')

In [ ]:
# Cellule 3.3 — Fonctions de génération et assemblage
TARGET_DURATION = 150  # 2min30 en secondes
CHUNK_SIZE = 10        # secondes par chunk

def p3_load_audio_mono(path: str, target_sr: int) -> np.ndarray:
    waveform, sr = torchaudio.load(path)
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    return waveform.squeeze().numpy()

def p3_save_tmp(audio_np: np.ndarray, sr: int, path: str):
    norm = (audio_np / (np.max(np.abs(audio_np)) + 1e-9) * 32767).astype(np.int16)
    scipy.io.wavfile.write(path, sr, norm)

def p3_generate_chunk(context_np: np.ndarray, sr: int, chunk_dur: int) -> np.ndarray:
    max_tokens = int(chunk_dur * p3_model.config.audio_encoder.frame_rate)
    context = context_np[-(sr * 10):]
    inputs = p3_processor(audio=context, sampling_rate=sr, return_tensors='pt').to(device)
    inputs['max_new_tokens'] = max_tokens
    with torch.no_grad():
        out = p3_model.generate(**inputs)
    return out[0, 0].cpu().numpy()

def p3_complete_music(audio_path: str, progress_cb=None) -> dict:
    if not p3_load_model():
        return {'success': False, 'path': '', 'error': 'Impossible de charger le modèle'}

    sr = p3_model.config.audio_encoder.sampling_rate
    os.makedirs('./generated', exist_ok=True)

    try:
        extract = p3_load_audio_mono(audio_path, sr)
        extract_dur = len(extract) / sr
        missing = max(0, TARGET_DURATION - extract_dur)

        intro_dur = missing * 0.45
        outro_dur = missing * 0.55
        n_intro = max(0, int(np.ceil(intro_dur / CHUNK_SIZE)))
        n_outro = max(0, int(np.ceil(outro_dur / CHUNK_SIZE)))
        total_steps = n_intro + n_outro

        if progress_cb:
            progress_cb(0, total_steps,
                f'📐 Extrait : {extract_dur:.0f}s → cible : {TARGET_DURATION}s | '
                f'intro : {intro_dur:.0f}s ({n_intro} chunks) | '
                f'outro : {outro_dur:.0f}s ({n_outro} chunks)')

        # --- INTRO ---
        intro_chunks = []
        if n_intro > 0:
            context = extract.copy()
            for i in range(n_intro):
                dur = min(CHUNK_SIZE, max(1, int(intro_dur - i * CHUNK_SIZE)))
                if progress_cb:
                    progress_cb(i + 1, total_steps, f'🎵 Intro chunk {i+1}/{n_intro} ({dur}s)...')
                chunk = p3_generate_chunk(context, sr, dur)
                intro_chunks.append(chunk)
                context = chunk
            intro_audio = np.concatenate(intro_chunks[::-1])
        else:
            intro_audio = np.array([])

        # --- OUTRO ---
        outro_chunks = []
        if n_outro > 0:
            context = extract.copy()
            for i in range(n_outro):
                dur = min(CHUNK_SIZE, max(1, int(outro_dur - i * CHUNK_SIZE)))
                if progress_cb:
                    progress_cb(n_intro + i + 1, total_steps, f'🎵 Outro chunk {i+1}/{n_outro} ({dur}s)...')
                chunk = p3_generate_chunk(context, sr, dur)
                outro_chunks.append(chunk)
                context = chunk
            outro_audio = np.concatenate(outro_chunks)
        else:
            outro_audio = np.array([])

        # --- ASSEMBLAGE ---
        parts = [p for p in [intro_audio, extract, outro_audio] if len(p) > 0]
        full_audio = np.concatenate(parts)
        out_path = './generated/completed_music.wav'
        p3_save_tmp(full_audio, sr, out_path)
        actual_dur = len(full_audio) / sr

        if progress_cb:
            progress_cb(total_steps, total_steps,
                f'✅ Assemblage terminé : {actual_dur:.0f}s ({actual_dur/60:.1f}min)')

        return {'success': True, 'path': out_path, 'duration': actual_dur, 'error': ''}

    except Exception as e:
        return {'success': False, 'path': '', 'duration': 0, 'error': str(e)}

print('✅ Fonctions Partie 3 prêtes — cible 2min30, chunks 10s, modèle small')

---
## 🖥️ PARTIE 4 — Interface graphique unifiée

In [ ]:
# Cellule 4.1 — Installation UI
!pip install ipywidgets --quiet
print('✅ ipywidgets installé')

In [ ]:
# Cellule 4.2 — Interface graphique unifiée
import ipywidgets as widgets
from IPython.display import display, Audio, HTML, clear_output
from google.colab import files as colab_files

def card(content, color='#f0fff0', border='#4CAF50'):
    return f"<div style='background:{color}; padding:14px; border-radius:10px; border:1px solid {border}; margin:6px 0;'>{content}</div>"
def error_card(msg):
    return card(f'<b>❌ Erreur</b><br>{msg}', '#fff0f0', '#f44336')
def info_card(msg):
    return card(msg, '#f0f4ff', '#5c6bc0')
def sep():
    return widgets.HTML("<hr style='border:1px solid #ddd; margin:20px 0;'>")

header = widgets.HTML("""
<div style='background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
     padding:24px; border-radius:14px; margin-bottom:10px;'>
  <h1 style='color:#e94560; margin:0;'>🎵 Music Studio</h1>
  <p style='color:#aaa; margin:6px 0 0 0;'>Télécharge · Identifie · Complète en 2min30</p>
</div>
""")

def refresh_dropdowns(path=''):
    choices = p1_get_audio_files() or ['Aucun fichier']
    s2_file.options = choices
    s3_file.options = choices
    if path and os.path.basename(path) in choices:
        s2_file.value = os.path.basename(path)
        s3_file.value = os.path.basename(path)

# ── SECTION 1 ──
s1_title = widgets.HTML("<h3 style='color:#E1306C; margin-bottom:8px;'>① Télécharger un Reel / Short</h3>")
s1_hint = widgets.HTML("<span style='color:#555; font-size:13px;'>📌 Instagram Reels & YouTube Shorts / vidéos</span>")
s1_url = widgets.Text(
    placeholder='https://www.instagram.com/reel/XXX  ou  https://youtube.com/shorts/XXX',
    description='URL :',
    layout=widgets.Layout(width='620px'),
    style={'description_width': '50px'}
)
s1_fmt = widgets.ToggleButtons(
    options=[('🎵 MP3', 'mp3'), ('🔊 WAV', 'wav'), ('🎞️ MP4', 'mp4')],
    value='mp3', description='Format :',
    style={'description_width': '70px', 'button_width': '120px'}
)
s1_cookies = widgets.Text(
    placeholder='Chemin cookies.txt (optionnel — Reels privés uniquement)',
    description='Cookies :',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '70px'}
)
s1_btn = widgets.Button(description='⬇️ Télécharger', button_style='danger',
    layout=widgets.Layout(width='180px', height='38px'))
s1_upload_btn = widgets.Button(description='📁 Importer depuis ordi', button_style='info',
    layout=widgets.Layout(width='210px', height='38px'))
s1_out = widgets.Output()

def on_s1_download(b):
    with s1_out:
        clear_output()
        url = s1_url.value.strip()
        if not url:
            display(HTML(error_card("Merci d'entrer une URL."))); return
        src = '📸 Instagram' if 'instagram.com' in url else '▶️ YouTube'
        display(HTML(info_card(f'⏳ Téléchargement depuis {src}...')))
        res = p1_download(url, s1_fmt.value, s1_cookies.value.strip())
        clear_output()
        if res['success']:
            size_kb = os.path.getsize(res['path']) / 1024
            display(HTML(card(
                f"<b>✅ {src} — {res['title']}</b><br>"
                f"📁 {os.path.basename(res['path'])} · {size_kb:.0f} KB"
            )))
            if s1_fmt.value in ['mp3', 'wav']:
                display(Audio(res['path']))
            colab_files.download(res['path'])
            refresh_dropdowns(res['path'])
        else:
            display(HTML(error_card(res['error'])))

def on_s1_upload(b):
    with s1_out:
        clear_output()
        display(HTML(info_card('⏳ Sélectionne ton fichier...')))
        uploaded = colab_files.upload()
        if uploaded:
            os.makedirs('./downloads', exist_ok=True)
            for fname, data in uploaded.items():
                dest = f'./downloads/{fname}'
                with open(dest, 'wb') as f:
                    f.write(data)
                display(HTML(card(f'✅ <b>{fname}</b> importé !')))
                display(Audio(dest))
                refresh_dropdowns(dest)

s1_btn.on_click(on_s1_download)
s1_upload_btn.on_click(on_s1_upload)

# ── SECTION 2 ──
s2_title = widgets.HTML("<h3 style='color:#FF9800; margin-bottom:8px;'>② Identifier le titre & le style</h3>")
s2_info = widgets.HTML(info_card(
    '🎯 <b>AudD.io</b> pour le titre exact (clé gratuite sur <a href="https://dashboard.audd.io" target="_blank">dashboard.audd.io</a>) '
    '+ <b>2 modèles HuggingFace</b> pour le genre musical.<br>'
    '<span style="color:#888; font-size:12px;">Sans clé AudD, seuls les genres seront détectés. Total : ~2-4min.</span>'
))
s2_audd_key = widgets.Text(
    placeholder='Colle ta clé API AudD.io ici (optionnel)',
    description='AudD Key :',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '90px'}
)
s2_file = widgets.Dropdown(
    options=p1_get_audio_files() or ['Aucun fichier'],
    description='Fichier :',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '70px'}
)
s2_refresh = widgets.Button(description='🔄', layout=widgets.Layout(width='50px'))
s2_refresh.on_click(lambda b: setattr(s2_file, 'options', p1_get_audio_files() or ['Aucun fichier']))
s2_progress_bar = widgets.IntProgress(value=0, min=0, max=3,
    description='', layout=widgets.Layout(width='580px'))
s2_progress_label = widgets.HTML('')
s2_btn = widgets.Button(description='🔍 Identifier', button_style='warning',
    layout=widgets.Layout(width='160px', height='38px'))
s2_out = widgets.Output()

def on_s2_identify(b):
    with s2_out:
        clear_output()
        fname = s2_file.value
        if fname == 'Aucun fichier':
            display(HTML(error_card("Télécharge d'abord un fichier en étape ①."))); return
        path = f'./downloads/{fname}'
        audd_key = s2_audd_key.value.strip()
        s2_progress_bar.value = 0

        def progress_cb(step, total, msg):
            s2_progress_bar.max = max(total, 1)
            s2_progress_bar.value = step
            s2_progress_label.value = info_card(msg)

        results = p2_identify_all(path, audd_key=audd_key, progress_cb=progress_cb)
        clear_output()

        html = "<h4 style='color:#FF9800;'>🎯 Résultats d'identification</h4>"
        tr = results['title']
        if tr['success']:
            spotify_btn = f" &nbsp;<a href='{tr['spotify_url']}' target='_blank' style='color:#1DB954;'>▶ Spotify</a>" if tr.get('spotify_url') else ''
            apple_btn = f" &nbsp;<a href='{tr['apple_url']}' target='_blank' style='color:#fc3c44;'>🍎 Apple Music</a>" if tr.get('apple_url') else ''
            html += card(
                f"<b>🎵 {tr['title']}</b> — {tr['artist']}<br>"
                f"💿 {tr['album']} · 📅 {tr.get('release_date','?')}"
                f"{spotify_btn}{apple_btn}",
                '#f0fff0', '#4CAF50'
            )
        else:
            html += card(
                f"🎵 <b>Titre non identifié</b> — {tr['error']}<br>"
                f"<span style='color:#888; font-size:12px;'>Ajoute une clé AudD.io pour identifier le titre exact.</span>",
                '#fff8e1', '#FFC107'
            )

        html += "<br><b style='color:#555;'>🤖 Analyse des genres :</b>"
        for res in results['genres']:
            if res['success']:
                rows = ''.join([
                    f"<tr>"
                    f"<td style='padding:3px 10px;'>#{i+1}</td>"
                    f"<td style='padding:3px 10px;'><b>{r['label']}</b></td>"
                    f"<td style='padding:3px 10px; color:#4CAF50;'>{r['score']*100:.1f}%</td>"
                    f"<td style='padding:3px 10px;'><div style='background:#4CAF50; height:10px; width:{r['score']*200:.0f}px; border-radius:5px;'></div></td>"
                    f"</tr>"
                    for i, r in enumerate(res['top_results'])
                ])
                html += card(
                    f"<b>🤖 {res['model_name']}</b> — <span style='color:#888; font-size:12px;'>{res['description']}</span><br><br>"
                    f"<table style='border-collapse:collapse; width:100%;'>{rows}</table>",
                    '#f8f8ff', '#7c83e5'
                )
            else:
                html += card(
                    f"<b>🤖 {res['model_name']}</b> — ❌ {res['error']}",
                    '#fff0f0', '#f44336'
                )
        display(HTML(html))

s2_btn.on_click(on_s2_identify)

# ── SECTION 3 ──
s3_title = widgets.HTML("<h3 style='color:#6200ea; margin-bottom:8px;'>③ Compléter en 2min30</h3>")
s3_info = widgets.HTML(info_card(
    '🧩 <b>Comment ça marche ?</b><br>'
    "L'extrait est positionné au tiers de la durée totale. "
    "MusicGen génère l'intro et l'outro par chunks de <b>10s</b> enchaînés, "
    'puis tout est assemblé en un fichier WAV de <b>2min30</b>.<br>'
    '<span style="color:#888; font-size:12px;">⚠️ La génération peut prendre 5-10min selon la durée manquante.</span>'
))
s3_file = widgets.Dropdown(
    options=p1_get_audio_files() or ['Aucun fichier'],
    description='Fichier :',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '70px'}
)
s3_refresh = widgets.Button(description='🔄', layout=widgets.Layout(width='50px'))
s3_refresh.on_click(lambda b: setattr(s3_file, 'options', p1_get_audio_files() or ['Aucun fichier']))
s3_progress_bar = widgets.IntProgress(value=0, min=0, max=100,
    description='', layout=widgets.Layout(width='580px'))
s3_progress_label = widgets.HTML('')
s3_btn = widgets.Button(description='🎼 Générer 2min30', button_style='success',
    layout=widgets.Layout(width='200px', height='42px'))
s3_out = widgets.Output()

def on_s3_generate(b):
    with s3_out:
        clear_output()
        fname = s3_file.value
        if fname == 'Aucun fichier':
            display(HTML(error_card("Télécharge d'abord un fichier en étape ①."))); return
        path = f'./downloads/{fname}'
        s3_progress_bar.value = 0
        s3_progress_label.value = info_card('⏳ Chargement du modèle MusicGen small...')

        def progress_cb(step, total, msg):
            s3_progress_bar.max = max(total, 1)
            s3_progress_bar.value = step
            s3_progress_label.value = info_card(msg)

        res = p3_complete_music(path, progress_cb=progress_cb)
        if res['success']:
            dur = res['duration']
            size_kb = os.path.getsize(res['path']) / 1024
            display(HTML(card(
                f"<b>✅ Musique complétée !</b><br>"
                f"⏱️ Durée finale : <b>{dur:.0f}s ({dur/60:.1f}min)</b><br>"
                f"📦 Taille : {size_kb:.0f} KB"
            )))
            display(Audio(res['path']))
            colab_files.download(res['path'])
        else:
            display(HTML(error_card(res['error'])))

s3_btn.on_click(on_s3_generate)

# ── ASSEMBLAGE FINAL ──
ui = widgets.VBox([
    header,
    s1_title, s1_hint,
    s1_url, s1_fmt, s1_cookies,
    widgets.HBox([s1_btn, widgets.HTML('&nbsp;&nbsp;'), s1_upload_btn]),
    s1_out,
    sep(),
    s2_title, s2_info,
    s2_audd_key,
    widgets.HBox([s2_file, s2_refresh]),
    s2_progress_bar, s2_progress_label,
    s2_btn,
    s2_out,
    sep(),
    s3_title, s3_info,
    widgets.HBox([s3_file, s3_refresh]),
    s3_progress_bar, s3_progress_label,
    s3_btn,
    s3_out,
], layout=widgets.Layout(padding='16px', width='720px'))

display(ui)